# 实验四 · 多层感知机识别 MNIST

依赖安装见[实验报告](README.md)。在 JupyterLab 中选择已安装依赖的 Python 内核，重启并运行全部单元格。

## 1. 导入依赖，设置路径和超参数

默认训练 10 轮，自动选择 CUDA 或 CPU。首次运行需要联网下载 MNIST。

In [ ]:
import json
from datetime import datetime
from pathlib import Path
import platform

import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

root = Path.cwd()
if root.name != "4th.exp":
    root = root / "4th.exp"
if not (root / "code.ipynb").is_file():
    raise FileNotFoundError("请从仓库根目录或 4th.exp 目录运行 Notebook。")

seed = 42
batch_size = 2048
epochs = 10
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Python {platform.python_version()} | PyTorch {torch.__version__} | {device}")

## 2. 加载数据并查看样例

`ToTensor()` 将像素缩放到 [0, 1]。两个模型使用同一训练集和测试集。

In [ ]:
transform = transforms.ToTensor()
train_data = datasets.MNIST(root / "data", train=True, download=True, transform=transform)
test_data = datasets.MNIST(root / "data", train=False, download=True, transform=transform)
test_loader = DataLoader(test_data, batch_size=batch_size, shuffle=False)
print("训练集", tuple(train_data.data.shape), "测试集", tuple(test_data.data.shape))

fig, axes = plt.subplots(1, 8, figsize=(10, 2))
for index, ax in enumerate(axes):
    image, label = train_data[index]
    ax.imshow(image.squeeze(), cmap="gray")
    ax.set_title(str(label))
    ax.axis("off")
fig.tight_layout()
plt.show()

## 3. 定义基础模型和改进模型

输入展平为 784 维。模型输出 10 个 logits，直接传给交叉熵损失。

In [ ]:
def baseline_model():
    return nn.Sequential(
        nn.Flatten(),
        nn.Linear(784, 128),
        nn.ReLU(),
        nn.Linear(128, 10),
    )


def improved_model():
    return nn.Sequential(
        nn.Flatten(),
        nn.Linear(784, 256),
        nn.BatchNorm1d(256),
        nn.GELU(),
        nn.Dropout(0.3),
        nn.Linear(256, 128),
        nn.BatchNorm1d(128),
        nn.GELU(),
        nn.Dropout(0.3),
        nn.Linear(128, 64),
        nn.BatchNorm1d(64),
        nn.GELU(),
        nn.Linear(64, 10),
    )

## 4. 定义训练与测试流程

平均损失按样本数加权。测试时停用 Dropout、使用 BatchNorm 的运行统计量，并关闭梯度计算。混淆矩阵的行表示真实类别，列表示预测类别。

In [ ]:
def train_one_epoch(model, loader, optimizer):
    model.train()
    loss_sum = 0.0
    correct = 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        logits = model(images)
        loss = nn.functional.cross_entropy(logits, labels)
        loss.backward()
        optimizer.step()
        loss_sum += loss.item() * labels.numel()
        correct += (logits.argmax(dim=1) == labels).sum().item()
    return loss_sum / len(loader.dataset), correct / len(loader.dataset)


@torch.inference_mode()
def evaluate(model, loader):
    model.eval()
    loss_sum = 0.0
    matrix = torch.zeros(10, 10, dtype=torch.int64)
    for images, labels in loader:
        logits = model(images.to(device))
        loss_sum += nn.functional.cross_entropy(logits, labels.to(device), reduction="sum").item()
        predicted = logits.argmax(dim=1).cpu()
        matrix += torch.bincount(10 * labels + predicted, minlength=100).reshape(10, 10)
    sample_count = len(loader.dataset)
    return loss_sum / sample_count, matrix.trace().item() / sample_count, matrix

## 5. 依次训练与测试

两组均从新模型开始，使用同样的批大小、轮数和训练样本顺序。基础模型用 SGD，改进模型用 Adam。

In [ ]:
experiments = (
    ("baseline", baseline_model, torch.optim.SGD, 0.1),
    ("improved", improved_model, torch.optim.Adam, 0.001),
)
results = {}
matrices = {}
for name, build_model, optimizer_type, lr in experiments:
    torch.manual_seed(seed)
    train_loader = DataLoader(
        train_data, batch_size=batch_size, shuffle=True,
        generator=torch.Generator().manual_seed(seed),
    )
    model = build_model().to(device)
    optimizer = optimizer_type(model.parameters(), lr=lr)
    history = []
    for epoch in range(1, epochs + 1):
        loss, accuracy = train_one_epoch(model, train_loader, optimizer)
        history.append({"epoch": epoch, "loss": loss, "accuracy": accuracy})
        print(f"{name:8} | {epoch:02}/{epochs} | loss={loss:.4f} | accuracy={accuracy:.2%}")

    test_loss, test_accuracy, matrix = evaluate(model, test_loader)
    assert matrix.sum().item() == len(test_data)
    matrices[name] = matrix
    results[name] = {
        "optimizer": optimizer_type.__name__,
        "learning_rate": lr,
        "parameters": sum(parameter.numel() for parameter in model.parameters()),
        "history": history,
        "test_loss": test_loss,
        "test_accuracy": test_accuracy,
        "test_correct": matrix.trace().item(),
        "confusion_matrix": matrix.tolist(),
    }
    print(f"{name:8} | test loss={test_loss:.4f} | test accuracy={test_accuracy:.2%}")

## 6. 绘图、保存实测结果

每次运行写入一个新的 `result/` 批次。报告可引用该批次的图片和 `metrics.json`。

In [ ]:
run_dir = root / "result" / datetime.now().strftime("%Y%m%d-%H%M%S-%f")
run_dir.mkdir(parents=True)

fig, (loss_ax, accuracy_ax) = plt.subplots(1, 2, figsize=(10, 4))
for name, result in results.items():
    history = result["history"]
    steps = [row["epoch"] for row in history]
    loss_ax.plot(steps, [row["loss"] for row in history], marker="o", label=name)
    accuracy_ax.plot(steps, [row["accuracy"] * 100 for row in history], marker="o", label=name)
loss_ax.set(xlabel="Epoch", ylabel="Training loss")
accuracy_ax.set(xlabel="Epoch", ylabel="Training accuracy (%)")
for ax in (loss_ax, accuracy_ax):
    ax.legend()
    ax.grid(alpha=0.25)
fig.tight_layout()
fig.savefig(run_dir / "training_curves.png", dpi=160)
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(12, 5), layout="constrained")
maximum_count = max(matrix.max().item() for matrix in matrices.values())
for ax, (name, matrix) in zip(axes, matrices.items()):
    plot = ax.imshow(matrix.numpy(), cmap="Blues", vmin=0, vmax=maximum_count)
    accuracy = results[name]["test_accuracy"]
    ax.set(title=f"{name} ({accuracy:.2%})", xlabel="Predicted digit", ylabel="True digit",
           xticks=range(10), yticks=range(10))
    for row in range(10):
        for column in range(10):
            count = matrix[row, column].item()
            color = "white" if count > maximum_count / 2 else "black"
            ax.text(column, row, str(count), ha="center", va="center", color=color, fontsize=7)
fig.colorbar(plot, ax=axes, shrink=0.8, label="Samples")
fig.savefig(run_dir / "confusion_matrices.png", dpi=160)
plt.show()

fig, ax = plt.subplots(figsize=(8, 4), layout="constrained")
for name, matrix in matrices.items():
    errors = matrix.sum(dim=1) - matrix.diag()
    ax.plot(range(10), 100 * errors.numpy() / matrix.sum(dim=1).numpy(), marker="o", label=name)
ax.set(xlabel="True digit", ylabel="Error rate (%)", xticks=range(10))
ax.legend()
ax.grid(alpha=0.25)
fig.savefig(run_dir / "per_class_error_rates.png", dpi=160)
plt.show()

metrics = {
    "python": platform.python_version(),
    "torch": str(torch.__version__),
    "device": str(device),
    "seed": seed,
    "batch_size": batch_size,
    "epochs": epochs,
    "train_samples": len(train_data),
    "test_samples": len(test_data),
    "models": results,
}
(run_dir / "metrics.json").write_text(json.dumps(metrics, indent=2), encoding="utf-8")
gain = 100 * (results["improved"]["test_accuracy"] - results["baseline"]["test_accuracy"])
print(f"测试准确率差值（改进 - 基础）: {gain:+.2f} 个百分点")
print(f"输出目录: {run_dir}")